# Cellpose on downscaled images

In [ ]:
import os
from pathlib import Path
from sphero_vem.preprocessing import imread_labels_downscaled
from dotenv import load_dotenv
import cellpose.metrics as metrics
import numpy as np
import tifffile
import pandas as pd
import matplotlib.pyplot as plt
from skimage.measure import regionprops_table
from matplotlib.ticker import ScalarFormatter
import scipy.stats as stats

load_dotenv("../.env")
DATA_ROOT = Path(os.environ.get("DATA_ROOT"))
dir_labeled = DATA_ROOT / "processed/labeled/Au_01-vol_01/labeled-01"
dir_output = DATA_ROOT / "processed/segmented/Au_01-vol_01/downscaling_tests"
dir_save = DATA_ROOT / "figures/segmentation"
dir_save.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "Helvetica"

In [ ]:
# Function definitions
def imwrite(fname: Path, image: np.ndarray) -> None:
    """Wrapper function to save images with common default options"""
    tifffile.imwrite(
        fname,
        image,
        compression="deflate",
        compressionargs={"level": 6},
        predictor=2,
        tile=(256, 256),
    )

In [ ]:
factors = [1, 2, 4, 5, 8, 10, 16, 20, 32]
avg_precision = []
precision_detail = []
for downscale_factor in factors:
    image_path = dir_labeled / "Au_01-vol_01-z_0425.tif"
    label_path = dir_labeled / f"labels/{image_path.stem}-cells.tif"
    seg_path = dir_output / f"cellposeSAM-pretrained-mask-ds{downscale_factor}.tif"

    labels = imread_labels_downscaled(label_path, downscale_factor)
    seg = tifffile.imread(seg_path)

    if downscale_factor == 1:
        regions = regionprops_table(labels, properties=["equivalent_diameter_area"])
        diameter = regions["equivalent_diameter_area"].mean()

    metric = metrics.average_precision(labels, seg, threshold=[0.5, 0.75])
    avg_precision.append((downscale_factor, metric[0][0], metric[0][1]))
    precision_detail.append(
        (downscale_factor, metric[0][0], metric[1][0], metric[2][0], metric[3][0])
    )

avg_precision_df = pd.DataFrame(
    avg_precision, columns=["downscale_factor", "AP@0.5", "AP@0.75"]
)
precision_detail_df = pd.DataFrame(
    precision_detail, columns=["downscale_factor", "AP", "tp", "fp", "fn"]
)

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax.plot(avg_precision_df["downscale_factor"], avg_precision_df["AP@0.5"])
ax.plot(avg_precision_df["downscale_factor"], avg_precision_df["AP@0.75"])
ax.set(
    xlim=[1, 32],
    ylim=[0, 1],
)
# Add labels
ax.set_xlabel("Downscale Factor")
ax.set_ylabel("Average Precision")
# ax.set_title('Average Precision vs Downscale Factor')
ax.legend(["AP@0.5 IoU", "AP@0.75 IoU"])

ax2 = ax.twinx()
ax2.plot(np.linspace(1, 32), diameter / np.linspace(1, 32), "r--")
ax2.set(ylim=[0, 1450])
ax2.set_ylabel("Average cell diameter [pixels]", color="red")
ax2.tick_params(axis="y", labelcolor="red")
# ax2.set_yscale("log")

# Add reference line at 120 pixels for ax2
ax2.axhspan(ymax=120, ymin=7.5, color="0.7", alpha=0.5)
# ax2.axhline(y=120, color='red', linestyle=':', alpha=0.7)
ax2.text(
    31,
    120 * 1.1,
    "Cell diameter range during training",
    color="red",
    ha="right",
    va="bottom",
)

# Add grid for better readability
ax.grid(True, linestyle="--", alpha=0.7)

fig.savefig(dir_save / "cellposeSAM-pretrained-precision-dowscaling.png", dpi=200)

In [ ]:
precision_detail_df

In [ ]:
# Timing
timing_df = pd.read_csv(dir_output / "cellposeSAM-downscaled-time.csv")

fig, ax = plt.subplots(layout="constrained", figsize=(6, 5))
ax.plot(1 / timing_df["downscale_factor"], timing_df["time"], "o")
ax.set(
    xlabel="Scaling factor",
    ylabel="Inference time [s]",
    xlim=[
        (1 / timing_df["downscale_factor"]).min() * 0.8,
        (1 / timing_df["downscale_factor"]).max() * 1.2,
    ],
    # ylim=[0, 180],
)
ax.set_yscale("log")
ax.set_xscale("log")
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_major_formatter(ScalarFormatter())

# # Add linear regression line

# Convert to log scale for regression
log_x = np.log10(1 / timing_df["downscale_factor"])
log_y = np.log10(timing_df["time"])

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(log_x, log_y)

# Plot regression line
x_range = np.logspace(
    np.log10((1 / timing_df["downscale_factor"]).min() * 0.8),
    np.log10((1 / timing_df["downscale_factor"]).max() * 1.2),
    100,
)
y_pred = 10 ** (intercept) * x_range ** (slope)
ax.plot(x_range, y_pred, "--", color="0.5", label=f"Slope: {slope:.2f}")

# Add text annotation with R²
# ax.text(0.05, 0.95, f'$R^2 = {r_value**2:.4f}$', transform=ax.transAxes,
#         bbox=dict(facecolor='white', alpha=0.7))
ax.legend()

ax.grid(True, linestyle="--", alpha=0.7)
fig.savefig(dir_save / "cellposeSAM-pretrained-time-dowscaling.png", dpi=200)

In [ ]:
cell_count = 0
nuclei_count = 0
for label_path in (dir_labeled / "labels").glob("*.tif"):
    labels = tifffile.imread(label_path)
    if "cells" in label_path.name:
        cell_count += labels.max()
    elif "nuclei" in label_path.name:
        nuclei_count += labels.max()

print(f"Cell count: {cell_count}")
print(f"Nuclei count: {nuclei_count}")